In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import random
%matplotlib inline
%matplotlib ipympl
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

In [7]:
data = pd.read_csv("data.csv")

print("DATA columns:")
print(data.columns.tolist())


DATA columns:
['order_id', 'user_id', 'status', 'gender', 'created_at', 'returned_at', 'shipped_at', 'delivered_at', 'num_of_item', 'product_id', 'inventory_item_id', 'sale_price', 'id', 'first_name', 'last_name', 'email', 'age', 'state', 'street_address', 'postal_code', 'city', 'country', 'traffic_source', 'user_geom', 'cost', 'category', 'brand', 'retail_price', 'department', 'sku', 'distribution_center_id', 'sold_at', 'product_category', 'product_name', 'product_brand', 'product_retail_price', 'product_department', 'product_sku', 'product_distribution_center_id', 'distribution_center_geom', 'order_item_id', 'delivery_longitude', 'delivery_latitude', 'warehouse_name', 'warehouse_longitude', 'warehouse_latitude', 'is_loyal', 'product_name_clean', 'customer_review']


In [8]:
data['status'].unique()

array(['Shipped', 'Complete', 'Cancelled', 'Returned', 'Processing'],
      dtype=object)

In [13]:
import pandas as pd


class SimplePreprocessor:
    def __init__(self):
        self.columns = [
            'user_id',
            'order_id',
            'status',
            'category',
            'brand',
            'age',
            'is_loyal',
            'gender',
            'country',
            'cost',
            'sale_price',
            'created_at',
            'product_id',
            'product_name_clean'
        ]

        self.valid_status = [
            'Shipped',
            'Complete',
            'Cancelled',
            'Returned',
            'Processing'
        ]

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        print(f"Initial rows: {len(df)}")

        # 1. оставить только нужные колонки
        df = df[self.columns]
        print(f"After column selection: {len(df)}")

        # 2. удалить дубликаты
        df = df.drop_duplicates()
        print(f"After drop_duplicates: {len(df)}")

        # 3. удалить строки с NaN
        df = df.dropna()
        print(f"After dropna: {len(df)}")

        # 4. фильтр статусов
        df = df[df['status'].isin(self.valid_status)]
        print(f"After status filter: {len(df)}")

        # 5. приведение типов
        df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
        df['age'] = pd.to_numeric(df['age'], errors='coerce')
        df['cost'] = pd.to_numeric(df['cost'], errors='coerce')
        df['sale_price'] = pd.to_numeric(df['sale_price'], errors='coerce')

        # 6. удалить строки после ошибок преобразования
        df = df.dropna(subset=['created_at', 'age', 'cost', 'sale_price'])
        print(f"After type cleaning: {len(df)}")

        return df.reset_index(drop=True)

In [14]:
prep = SimplePreprocessor()

df_clean = prep.transform(data)

Initial rows: 545778
After column selection: 545778
After drop_duplicates: 181922
After dropna: 181731
After status filter: 181731
After type cleaning: 177292


In [15]:
import pandas as pd
import numpy as np
import os

df = df_clean.copy()

# =========================
# 1. ФИЛЬТР
# =========================
df = df[df['status'].isin(['Complete', 'Returned'])].copy()
df = df.dropna(subset=['gender', 'category', 'brand', 'product_id'])

# =========================
# ФУНКЦИЯ ДЛЯ ОДНОГО ПОЛА
# =========================
def build_tables(df_gender):

    df_gender = df_gender.copy()

    # -------------------------
    # МАРЖА
    # -------------------------
    df_gender['margin'] = df_gender['sale_price'] - df_gender['cost']

    # -------------------------
    # PRODUCT LEVEL
    # -------------------------
    prod = df_gender.groupby(
        ['category', 'brand', 'product_id', 'product_name_clean']
    ).agg(
        purchases=('status', lambda x: (x == 'Complete').sum()),
        avg_price=('sale_price', 'mean')
    ).reset_index()

    prod = prod.sort_values(
        ['category', 'brand', 'purchases', 'avg_price'],
        ascending=[True, True, False, False]
    )

    prod['rank'] = prod.groupby(['category', 'brand']).cumcount() + 1

    # -------------------------
    # БРЕНДЫ С >= 2 ТОВАРАМИ
    # -------------------------
    brand_counts = (
        prod.groupby(['category', 'brand'])['product_id']
        .nunique()
        .reset_index(name='n_products')
    )

    valid_brands = brand_counts[
        brand_counts['n_products'] >= 2
    ][['category', 'brand']]

    prod = prod.merge(valid_brands, on=['category', 'brand'], how='inner')

    # -------------------------
    # ТОП-2 ТОВАРА
    # -------------------------
    top_prod = prod[prod['rank'] <= 2].copy()

    top_prod = top_prod.pivot_table(
        index=['category', 'brand'],
        columns='rank',
        values=['product_id', 'product_name_clean', 'avg_price'],
        aggfunc='first'
    )

    top_prod.columns = [
        f"{col_name}_{rank}"
        for col_name, rank in top_prod.columns
    ]

    top_prod = top_prod.rename(columns={
        'product_id_1': 'product_1',
        'product_id_2': 'product_2',
        'product_name_clean_1': 'product_name_1',
        'product_name_clean_2': 'product_name_2',
        'avg_price_1': 'price_1',
        'avg_price_2': 'price_2'
    })

    top_prod = top_prod.reset_index()

    # -------------------------
    # BRAND LEVEL
    # -------------------------
    df_gender = df_gender.merge(valid_brands, on=['category', 'brand'], how='inner')

    agg = df_gender.groupby(['category', 'brand']).agg(
        complete_cnt=('status', lambda x: (x == 'Complete').sum()),
        returned_cnt=('status', lambda x: (x == 'Returned').sum()),
        avg_margin=('margin', 'mean'),
        avg_price=('sale_price', 'mean')
    ).reset_index()

    agg['purchases'] = agg['complete_cnt']

    # -------------------------
    # BUYOUT
    # -------------------------
    agg['buyout_rate'] = (
        agg['complete_cnt'] /
        (agg['complete_cnt'] + agg['returned_cnt'] + 1e-9)
    )

    global_rate = (
        (df_gender['status'] == 'Complete').sum() /
        len(df_gender)
    )

    agg['buyout_smooth'] = (
        agg['complete_cnt'] + 10 * global_rate
    ) / (
        agg['complete_cnt'] + agg['returned_cnt'] + 10
    )

    # -------------------------
    # ДОП. МЕТРИКИ
    # -------------------------
    agg['expected_margin'] = agg['avg_margin'] * agg['buyout_smooth']
    agg['log_purchases'] = np.log1p(agg['purchases'])

    # -------------------------
    # НОРМАЛИЗАЦИЯ
    # -------------------------
    def normalize(s):
        return (s - s.min()) / (s.max() - s.min() + 1e-9)

    agg['margin_norm'] = normalize(agg['avg_margin'])
    agg['buyout_norm'] = normalize(agg['buyout_smooth'])
    agg['popularity_norm'] = normalize(agg['log_purchases'])

    # -------------------------
    # BOOST
    # -------------------------
    agg['boost'] = np.where(agg['purchases'] > 10, 1.1, 1.0)

    # -------------------------
    # ACTUAL SCORE
    # -------------------------
    agg['actual_score'] = (
        0.5 * agg['margin_norm'] +
        0.5 * agg['buyout_norm']
    ) * agg['boost']

    # -------------------------
    # RATING
    # -------------------------
    base = agg['buyout_smooth']

    agg['rating'] = 2 + 3 * (
        (base - base.min()) /
        (base.max() - base.min() + 1e-9)
    )

    agg['rating'] = agg['rating'].round(2)
    agg['rating_boosted'] = agg['rating'] * agg['boost']

    # -------------------------
    # MERGE С ТОП ТОВАРАМИ
    # -------------------------
    agg = agg.merge(top_prod, on=['category', 'brand'], how='left')

    # -------------------------
    # CLEAN UI
    # -------------------------
    for col in ['product_name_1', 'product_name_2']:
        if col in agg.columns:
            agg[col] = agg[col].fillna('')

    # -------------------------
    # СОРТИРОВКИ
    # -------------------------
    by_price_desc = agg.sort_values(
        ['category', 'avg_price'],
        ascending=[True, False]
    )

    by_price_asc = agg.sort_values(
        ['category', 'avg_price'],
        ascending=[True, True]
    )

    by_popularity = agg.sort_values(
        ['category', 'purchases', 'boost'],
        ascending=[True, False, False]
    )

    by_rating = agg.sort_values(
        ['category', 'rating_boosted'],
        ascending=[True, False]
    )

    by_actual = agg.sort_values(
        ['category', 'actual_score'],
        ascending=[True, False]
    )

    return {
        'price_desc': by_price_desc.reset_index(drop=True),
        'price_asc': by_price_asc.reset_index(drop=True),
        'popularity': by_popularity.reset_index(drop=True),
        'rating': by_rating.reset_index(drop=True),
        'actual': by_actual.reset_index(drop=True)
    }

# =========================
# 2. РАЗДЕЛЕНИЕ ПО ПОЛУ
# =========================
df_male = df[df['gender'] == 'M'].copy()
df_female = df[df['gender'] == 'F'].copy()

male_tables = build_tables(df_male)
female_tables = build_tables(df_female)

# =========================
# 3. СОХРАНЕНИЕ
# =========================
os.makedirs('results', exist_ok=True)

for sort_name, table in male_tables.items():
    table.to_csv(f'results/male_{sort_name}.csv', index=False)

for sort_name, table in female_tables.items():
    table.to_csv(f'results/female_{sort_name}.csv', index=False)

# =========================
# RESULT
# =========================
print(male_tables['actual'].head(10))
print(female_tables['actual'].head(10))

      category            brand  complete_cnt  returned_cnt  avg_margin  \
0  Accessories         Tom Ford             8             1  171.383088   
1  Accessories          JiMarti            14             1   16.581424   
2  Accessories     K. Alexander            14             1    4.950690   
3  Accessories            Magic            14             1    3.201816   
4  Accessories        SmartWool             9             0   23.560289   
5  Accessories         Electric             7             0   35.901815   
6  Accessories    Wurkin Stiffs             8             0   23.520000   
7  Accessories  Harley-Davidson            35             7   12.710191   
8  Accessories         Carhartt            64            15   14.579361   
9  Accessories         Isotoner            23             5   17.927171   

    avg_price  purchases  buyout_rate  buyout_smooth  expected_margin  ...  \
0  268.250007          8     0.888889       0.797346       136.651653  ...   
1   27.083334     

In [16]:
import os

os.makedirs("results", exist_ok=True)

# =========================
# MALE
# =========================
for name, table in male_tables.items():
    table.to_csv(f"results/male_{name}.csv", index=False)

# =========================
# FEMALE
# =========================
for name, table in female_tables.items():
    table.to_csv(f"results/female_{name}.csv", index=False)

In [17]:
import pandas as pd
import os

def recommend_brands(
    gender: str,
    category: str,
    n: int = 5,
    sort_type: str = "actual",
    path: str = "results",
    full: bool = False
):
    """
    Рекомендация брендов по категории и полу.
    """

    gender_map = {
        'M': 'male',
        'F': 'female'
    }

    if gender not in gender_map:
        raise ValueError("gender должен быть 'M' или 'F'")

    valid_sorts = [
        'actual',
        'price_desc',
        'price_asc',
        'popularity',
        'rating'
    ]

    if sort_type not in valid_sorts:
        raise ValueError(f"sort_type должен быть из {valid_sorts}")

    filepath = os.path.join(
        path,
        f"{gender_map[gender]}_{sort_type}.csv"
    )

    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Файл не найден: {filepath}")

    df = pd.read_csv(filepath)

    # =========================
    # ФИЛЬТР ПО КАТЕГОРИИ
    # =========================
    df = df[df['category'] == category].copy()

    if df.empty:
        return pd.DataFrame()

    # =========================
    # ТОП-N
    # =========================
    df = df.head(n)

    # =========================
    # ЧИСТКА NaN
    # =========================
    text_cols = [
        'brand',
        'product_1',
        'product_2',
        'product_name_1',
        'product_name_2'
    ]

    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].fillna('').astype(str)

    numeric_cols = [
        'rating',
        'price_1',
        'price_2'
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # =========================
    # SHORT MODE
    # =========================
    if not full:
        visible_cols = [
            'brand',
            'rating',

            'product_1',
            'product_name_1',
            'price_1',

            'product_2',
            'product_name_2',
            'price_2'
        ]

        visible_cols = [c for c in visible_cols if c in df.columns]
        df = df[visible_cols]

    return df.reset_index(drop=True)

In [18]:
recommend_brands(
    gender='F',
    category='Accessories',
    n=10,
    sort_type='actual',
    full=False
)

,brand,rating,product_1,product_name_1,price_1,product_2,product_name_2,price_2
0,Oakley,4.33,14173,Oakley Women's Unfaithful Rectangular Sunglasses,83.820000,14121,Oakley Women's Disclosure Oval Polarized Sungl...,109.000000
1,Carrera,4.49,13885,Carrera Topcar 1/S Aviator Sunglasses,85.419998,14256,Carrera Endurance T/S JO9 LF,80.949997
2,Coach,4.29,14069,Coach Classic Leather Turnlock Swingpack Cross...,130.000000,14238,Coach Julia Op Art Slim Zip Wallet 47379 (SV/B...,119.050003
3,Tom Ford,3.95,13758,Tom Ford Marko FT0144 Sunglasses - 18V Shiny R...,297.950012,14203,Tom Ford Jennifer FT0008 Sunglasses - 38F Tran...,214.500000
4,boxed-gifts,4.77,14245,100% Wool 'Hat-imals' Plush Knit Winter Hats (...,14.990000,13789,Boxed Fancy 3 pc Ladies Cotton Handkerchiefs,6.990000
5,Costa Del Mar,4.23,13932,Costa Del Mar Brine Polarized Sunglasses - Cos...,188.949997,14182,Costa Del Mar Caballito Polarized Sunglasses -...,218.949997
6,Gucci,4.14,14162,Gucci 3164 sunglasses,176.979996,13843,Sunglasses Gucci 4225/S 0ADJ White,239.949997
7,180s,4.61,13615,180s Women's Tahoe Earmuff,30.000000,13756,180s Women's Lush Ear Warmer,24.990000
8,HOBO,4.33,14026,Hobo Women's Belinda CP-4102BLK Wallet,87.940002,14250,Hobo Women's Millie VI-3978BOR Wallet,55.619999
9,Pashmina,4.67,14286,Plain Colour Design 100% Pashmina Shawl Scarf ...,29.990000,13915,Karakoram Design 100% Pashmina Shawl Scarf Wra...,45.990002
